# Lab 2: The "Scrub" Stage – Data Cleaning, Text Tokenization & Feature Extraction
**Course:** CIS 531/731 - Data Science and Analytics (Fall 2026)

**Objective:** Process raw ingested data through the **Scrub** phase of the OSEMN pipeline using PySpark SQL and PySpark ML. Execute missing-value handling, text normalization, stop-word removal, and vector feature extraction (TF-IDF).

## Part 1: Environment Setup & SparkSession Bring-Up

In [1]:
# Install PySpark if running in a cloud notebook environment
!pip install -q pyspark

In [2]:
import sys
import os
import pyspark
import subprocess

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("PySpark version:", pyspark.__version__)
print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))

subprocess.run(["java", "-version"])

Python executable: C:\Users\dacnu\anaconda3\python.exe
Python version: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
PySpark version: 4.0.4
PYSPARK_PYTHON: None


CompletedProcess(args=['java', '-version'], returncode=0)

In [3]:

import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, FloatType
)

spark = (
    SparkSession.builder
    .appName("Lab2_Scrub_Stage")
    .master("local[2]")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

print("Spark version:", spark.version)


Spark version: 4.0.1


In [4]:
test_df = spark.createDataFrame(
    [(1, "Hello Spark"), (2, "Testing Python worker")],
    ["id", "message"]
)

test_df.show()

+---+--------------------+
| id|             message|
+---+--------------------+
|  1|         Hello Spark|
|  2|Testing Python wo...|
+---+--------------------+



## Part 2: Data Cleaning & Quality Enforcements
In this section, we ingest raw multi-field records, filter out corrupt/null values, deduplicate entries, and apply regex-based string normalization.

In [5]:
# Define raw schema for incoming document batch
raw_schema = StructType([
    StructField("doc_id", StringType(), True),
    StructField("author", StringType(), True),
    StructField("category", StringType(), True),
    StructField("raw_text", StringType(), True)
])

# Sample raw dataset containing dirty/duplicate records, missing fields, and irregular casing/punctuation
raw_data = [
    ("doc_001", "Alice Smith", "AI", "  PySpark makes large-scale computational processing fast and flexible!  "),
    ("doc_002", "Bob Jones", "Database", "Vector databases store high-dimensional embeddings for fast retrieval."),
    ("doc_003", None, "AI", "Retrieval-augmented generation (RAG) enhances LLM factual reliability."),
    ("doc_001", "Alice Smith", "AI", "  PySpark makes large-scale computational processing fast and flexible!  "), # Duplicate
    ("doc_004", "Carol White", None, None), # Corrupt/Null text
    ("doc_005", "Dave Brown", "ASR", "Automatic Speech Recognition converts spoken audio streams into structured text.")
]

df_raw = spark.createDataFrame(raw_data, schema=raw_schema)
print("--- Raw Dataset ---")
df_raw.show(truncate=False)

--- Raw Dataset ---
+-------+-----------+--------+--------------------------------------------------------------------------------+
|doc_id |author     |category|raw_text                                                                        |
+-------+-----------+--------+--------------------------------------------------------------------------------+
|doc_001|Alice Smith|AI      |  PySpark makes large-scale computational processing fast and flexible!         |
|doc_002|Bob Jones  |Database|Vector databases store high-dimensional embeddings for fast retrieval.          |
|doc_003|NULL       |AI      |Retrieval-augmented generation (RAG) enhances LLM factual reliability.          |
|doc_001|Alice Smith|AI      |  PySpark makes large-scale computational processing fast and flexible!         |
|doc_004|Carol White|NULL    |NULL                                                                            |
|doc_005|Dave Brown |ASR     |Automatic Speech Recognition converts spoken audio str

In [6]:
# 1. Drop records missing critical attributes (doc_id or raw_text)
df_cleaned = df_raw.dropna(subset=["doc_id", "raw_text"])

# 2. Deduplicate records based on primary identifier
df_cleaned = df_cleaned.dropDuplicates(["doc_id"])

# 3. Fill non-critical missing metadata
df_cleaned = df_cleaned.fillna({"author": "Unknown", "category": "Uncategorized"})

# 4. Text Normalization: trim whitespace, convert to lowercase, remove special characters
df_scrubbed = df_cleaned.withColumn(
    "clean_text",
    F.regexp_replace(
        F.lower(F.trim(F.col("raw_text"))),
        r"[^a-z0-9\s-]", ""
)
)

print("--- Scrubbed Dataset ---")
df_scrubbed.select("doc_id", "author", "category", "clean_text").show(truncate=False)

--- Scrubbed Dataset ---
+-------+-----------+--------+-------------------------------------------------------------------------------+
|doc_id |author     |category|clean_text                                                                     |
+-------+-----------+--------+-------------------------------------------------------------------------------+
|doc_001|Alice Smith|AI      |pyspark makes large-scale computational processing fast and flexible           |
|doc_002|Bob Jones  |Database|vector databases store high-dimensional embeddings for fast retrieval          |
|doc_003|Unknown    |AI      |retrieval-augmented generation rag enhances llm factual reliability            |
|doc_005|Dave Brown |ASR     |automatic speech recognition converts spoken audio streams into structured text|
+-------+-----------+--------+-------------------------------------------------------------------------------+



## Part 3: NLP Pipeline – Tokenization & Stop-Words Removal
We utilize `pyspark.ml.feature` to tokenize normalized text and filter out uninformative stop words.

In [7]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover

# Step 1: Tokenize cleaned text into word arrays
tokenizer = Tokenizer(inputCol="clean_text", outputCol="raw_tokens")
df_tokenized = tokenizer.transform(df_scrubbed)

# Step 2: Remove common English stop words
remover = StopWordsRemover(inputCol="raw_tokens", outputCol="filtered_tokens")
df_filtered = remover.transform(df_tokenized)

print("--- Tokenized and Filtered Features ---")
df_filtered.select("doc_id", "filtered_tokens").show(truncate=False)

--- Tokenized and Filtered Features ---
+-------+------------------------------------------------------------------------------------+
|doc_id |filtered_tokens                                                                     |
+-------+------------------------------------------------------------------------------------+
|doc_001|[pyspark, makes, large-scale, computational, processing, fast, flexible]            |
|doc_002|[vector, databases, store, high-dimensional, embeddings, fast, retrieval]           |
|doc_003|[retrieval-augmented, generation, rag, enhances, llm, factual, reliability]         |
|doc_005|[automatic, speech, recognition, converts, spoken, audio, streams, structured, text]|
+-------+------------------------------------------------------------------------------------+



## Part 4: Feature Extraction – TF-IDF Vectorization
Convert sequence tokens into numerical term-frequency and inverse-document-frequency vectors for downstream analytical models or index construction.

In [8]:
from pyspark.ml.feature import HashingTF, IDF

# Step 1: Compute Term Frequency (TF) vectors
hashingTF = HashingTF(inputCol="filtered_tokens", outputCol="raw_features", numFeatures=20)
df_tf = hashingTF.transform(df_filtered)

# Step 2: Compute Inverse Document Frequency (IDF) weights
idf = IDF(inputCol="raw_features", outputCol="tfidf_features")
idfModel = idf.fit(df_tf)
df_tfidf = idfModel.transform(df_tf)

print("--- Final Feature Representations (TF-IDF) ---")
df_tfidf.select("doc_id", "filtered_tokens", "tfidf_features").show(truncate=False)

--- Final Feature Representations (TF-IDF) ---
+-------+------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|doc_id |filtered_tokens                                                                     |tfidf_features                                                                                                                                                                      |
+-------+------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|doc_001|[pyspark, makes, large-scale, computational, processing, fast, flexible]            |(20,[1,3,6,7,13,15],[1.02165124

## Part 5: Laboratory Verification & Deliverables Checklist

**Instructions:** Complete the following verification tasks for your Lab 2 assessment submission:

1. **Execution Output Screenshot:** Verify that all code blocks execute without error in local Spark mode and produce non-empty TF-IDF sparse vectors.
2. **Pipeline Integration Question:** Describe how your team's chosen Term Project Track (ASR or Vector DB) will integrate this Scrub stage prior to model training or vector store ingestion.
3. **Code Extension Task:** Modify the regex pattern in Part 2 to retain hyphenated compound words (e.g., `large-scale`) as single token units instead of splitting or stripping hyphens.